Prepare the environment

In [1]:
!pip install 'accelerate>=0.26.0'

In [2]:
!pip install transformers datasets kagglehub

Use tokenizer and model from google-t5-small

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Download Shakesperean Dataset which contains Shakesperean Text and their corresponding translations

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("garnavaurha/shakespearify")

print("Path to dataset files:", path)
path += "/final.csv"

Path to dataset files: /teamspace/studios/this_studio/.cache/kagglehub/datasets/garnavaurha/shakespearify/versions/1


Load the csv file as a dataset

In [5]:
from datasets import load_dataset

raw_dataset = load_dataset(
    "csv",
    data_files={
        "train": path
    }
)

raw_dataset = raw_dataset["train"]
raw_dataset

Dataset({
    features: ['Unnamed: 0', 'id', 'og', 't'],
    num_rows: 51787
})

Prepare the dataset for training

In [6]:
def preprocess_function(example):
    inputs = ["translate Modern English to Shakesperean English: " + str(x) for x in example['t']]
    targets = [str(x) for x in example['og']]

    model_data = tokenizer(inputs, max_length=512, padding="max_length")
    label_data = tokenizer(targets, max_length=512, padding="max_length")

    model_data["labels"] = label_data['input_ids']
    return model_data


tokenized_datasets = raw_dataset.map(preprocess_function, batched = True, remove_columns=raw_dataset.column_names)
tokenized_datasets


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 51787
})

Prepare data_collator for automatic handling of padding

In [7]:
from transformers import  DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


Initialize the Training Arguments and start training the model

In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "shakespeare-google-t5",
    learning_rate=5e-5,
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    save_total_limit=3,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
500,0.563700
1000,0.157000
1500,0.151200
2000,0.146500
2500,0.146300
3000,0.144200
3500,0.143700
4000,0.141600
4500,0.138400
5000,0.139500


TrainOutput(global_step=8095, training_loss=0.1682426801355184, metrics={'train_runtime': 1873.8532, 'train_samples_per_second': 138.183, 'train_steps_per_second': 4.32, 'total_flos': 3.504472936415232e+16, 'train_loss': 0.1682426801355184, 'epoch': 5.0})

Test the model for a few examples

In [ ]:
def speak_shakespeare(text):
    input_text = "translate modern English to Shakespearean English: " + text
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(speak_shakespeare("The risk is too much for me to handle."))
print(speak_shakespeare("I love you"))
print(speak_shakespeare("What are you doing today?"))
print(speak_shakespeare("You foolish thing."))

The risk is too much for me to handle.
I love thee
What dost thou today?
You foolish thing.
